In [1]:
import json
import re
import os
import pandas as pd
from collections import Counter, defaultdict
from pprint import pprint

BASE = 'LLM_Redial/Movie'

with open(f'{BASE}/item_map.json') as f:
    item_map = json.load(f)

with open(f'{BASE}/user_ids.json') as f:
    user_ids = json.load(f)

with open(f'{BASE}/final_data.jsonl') as f:
    users_raw = [json.loads(line) for line in f]

with open(f'{BASE}/Conversation.txt', encoding='utf-8') as f:
    raw_conversations = f.read()

print(f'Movies: {len(item_map)} | Users: {len(users_raw)} | User IDs: {len(user_ids)} | Conv chars: {len(raw_conversations):,}')

Movies: 9687 | Users: 3131 | User IDs: 3131 | Conv chars: 16,935,069


In [4]:
# quick look at item_map
sample = list(item_map.items())[:10]
pprint(sample)

[('630352138X', 'No Highway In The Sky VHS'),
 ('630434063X', 'Outland VHS'),
 ('B00006SFIW', 'The Black Raven'),
 ('6303142346', 'La Bella y La Bestia Beauty and the Beast  VHS'),
 ('B001UHOWX8', 'White Christmas'),
 ('6301977343', 'Silk Stockings VHS'),
 ('B00005IBK9', 'Captain Scarlet and the Mysterons'),
 ('6302215927', 'Andersonville Trial VHS'),
 ('B00005JL8Z', 'The Tuxedo'),
 ('B00004YS6O', 'Lonely Wives')]


In [5]:
# quick look at one user profile
pprint(users_raw[0])

{'A30Q8X8B1S3GGT': {'Conversation': [{'conversation_1': {'conversation_id': 0,
                                                         'rec_item': ['B0001DYG38'],
                                                         'user_dislikes': [],
                                                         'user_likes': ['B001UHOWX8']}},
                                     {'conversation_2': {'conversation_id': 1,
                                                         'rec_item': ['B000N3T0GO'],
                                                         'user_dislikes': ['B00005A07V'],
                                                         'user_likes': []}}],
                    'history_interaction': ['630352138X',
                                            '630434063X',
                                            'B00006SFIW',
                                            '6303142346',
                                            'B001UHOWX8',
                                            '63

In [6]:
# parse all user profiles into a flat structure
profiles = []
for entry in users_raw:
    uid, info = next(iter(entry.items()))
    profiles.append({
        'user_id': uid,
        'history_count': len(info['history_interaction']),
        'might_like_count': len(info['user_might_like']),
        'conv_count': len(info['Conversation']),
        'info': info,
    })

df_users = pd.DataFrame([{k: v for k, v in p.items() if k != 'info'} for p in profiles])
df_users.describe()

,history_count,might_like_count,conv_count
count,3131.000000,3131.000000,3131.000000
mean,47.444267,3.222293,3.222293
std,64.087228,4.760235,4.760235
min,8.000000,1.000000,1.000000
25%,18.000000,1.000000,1.000000
50%,30.000000,2.000000,2.000000
75%,52.000000,3.000000,3.000000
max,960.000000,87.000000,87.000000


In [7]:
df_users.head(5)

,user_id,history_count,might_like_count,conv_count
0,A30Q8X8B1S3GGT,19,2,2
1,A2NBOL825B93OM,38,3,3
2,A1HO9J4DCQDGP9,46,3,3
3,A1EMDSTJDUE6B0,120,9,9
4,A23QOAXJSWIBS6,41,1,1


In [8]:
# movie name stats
names = list(item_map.values())
lengths = [len(n) for n in names]
print(f'Shortest: {min(lengths)} chars -> {names[lengths.index(min(lengths))]}')
print(f'Longest:  {max(lengths)} chars -> {names[lengths.index(max(lengths))][:100]}...')
print(f'Avg: {sum(lengths)/len(lengths):.1f} chars')

vhs = sum(1 for n in names if 'VHS' in n)
dvd = sum(1 for n in names if 'DVD' in n)
bluray = sum(1 for n in names if 'Blu' in n)
print(f'\nVHS: {vhs} | DVD: {dvd} | Blu-ray: {bluray}  (out of {len(names)})')

Shortest: 1 chars -> O
Longest:  352 chars -> Nausicaa of the Valley of the Wind (1985) / REGION 6 DVD / Audio: English, Japanese, French, Simp. a...
Avg: 24.2 chars

VHS: 2927 | DVD: 167 | Blu-ray: 128  (out of 9687)


In [9]:
# conversation metadata stats
total_likes = total_dislikes = total_recs = 0
conv_rows = []

for p in profiles:
    for j, conv in enumerate(p['info']['Conversation']):
        cd = conv[f'conversation_{j+1}']
        nl = len(cd['user_likes'])
        nd = len(cd['user_dislikes'])
        total_likes += nl
        total_dislikes += nd
        total_recs += len(cd['rec_item'])
        conv_rows.append({
            'user_id': p['user_id'],
            'conv_id': cd['conversation_id'],
            'n_likes': nl,
            'n_dislikes': nd,
            'n_recs': len(cd['rec_item']),
            'liked_ids': cd['user_likes'],
            'disliked_ids': cd['user_dislikes'],
            'rec_ids': cd['rec_item'],
        })

df_convs = pd.DataFrame(conv_rows)
print(f'Total convos: {len(df_convs)} | Likes: {total_likes} | Dislikes: {total_dislikes} | Recs: {total_recs}')
df_convs[['n_likes', 'n_dislikes', 'n_recs']].describe()

Total convos: 10089 | Likes: 14064 | Dislikes: 7982 | Recs: 10089


,n_likes,n_dislikes,n_recs
count,10089.000000,10089.000000,10089.0
mean,1.393993,0.791159,1.0
std,1.036453,0.984161,0.0
min,0.000000,0.000000,1.0
25%,1.000000,0.000000,1.0
50%,1.000000,0.000000,1.0
75%,2.000000,1.000000,1.0
max,3.000000,3.000000,1.0


In [10]:
# most watched movies
all_history = []
for p in profiles:
    all_history.extend(p['info']['history_interaction'])

movie_freq = Counter(all_history)
top20_watched = [(item_map.get(mid, 'UNK'), cnt) for mid, cnt in movie_freq.most_common(20)]
pd.DataFrame(top20_watched, columns=['Movie', 'Watch_Count'])

,Movie,Watch_Count
0,The Lord of the Rings: The Fellowship of the Ring,199
1,"Terminator, The",187
2,Batman Begins,174
3,The Sixth Sense VHS,173
4,THE SIXTH SENSE,172
5,Kill Bill: Volume 1,170
6,Pirates of the Caribbean: The Curse of the Bla...,164
7,The Lord of the Rings: The Return of the King,161
8,Bubbe Meises Bubbe Stories VHS,161
9,Star Wars Trilogy THX Digitally Mastered Edition,160


In [11]:
# most liked movies in conversations
all_likes = []
for row in conv_rows:
    all_likes.extend(row['liked_ids'])

like_freq = Counter(all_likes)
top20_liked = [(item_map.get(mid, 'UNK'), cnt) for mid, cnt in like_freq.most_common(20)]
pd.DataFrame(top20_liked, columns=['Movie', 'Like_Count'])

,Movie,Like_Count
0,The Lord of the Rings: The Return of the King,29
1,"Terminator, The",28
2,The Lord of the Rings: The Fellowship of the Ring,26
3,The Sixth Sense VHS,25
4,Batman Begins,21
5,The Silence of the Lambs VHS,20
6,Gladiator VHS,19
7,Finding Nemo (Mandarin Chinese Edition) [2 DVDs],19
8,The Others,19
9,Lord of the Rings Return of the King 2 Disc Ex...,18


In [12]:
# parse Conversation.txt into {id: dialogue}
def parse_conversations(text):
    convos = {}
    blocks = text.strip().split('\n\n')
    current_id = None
    current_lines = []
    for block in blocks:
        block = block.strip()
        if block.isdigit():
            if current_id is not None:
                convos[current_id] = '\n\n'.join(current_lines).strip()
            current_id = int(block)
            current_lines = []
        else:
            current_lines.append(block)
    if current_id is not None:
        convos[current_id] = '\n\n'.join(current_lines).strip()
    return convos

convos = parse_conversations(raw_conversations)
print(f'Parsed: {len(convos)} conversations | IDs: {min(convos)}..{max(convos)}')

Parsed: 10089 conversations | IDs: 0..10088


In [13]:
# sample conversation
print(convos[0])

User: Hi, I'm Mark Savary. I really enjoyed watching "White Christmas". It's a semi-remake of "Holiday Inn" and features Bing Crosby and Fred Astaire. Bing and Danny Kaye play ex-soldiers who become show business sensations after World War II. They meet the Haynes sisters, Vera-Ellen and Rosemary Clooney, and they all end up in Vermont to put on a show at an inn run by their ex-Army General. It's a delightful family movie with great performances and catchy tunes like "Snow" and "Count Your Troubles".

Agent: Oh, I completely agree! "White Christmas" is a wonderful film that you can watch over and over again. Bing Crosby, Rosemary Clooney, and Danny Kaye deliver fantastic performances. The story, the songs, everything about it is just amazing. By the way, have you seen "Never a Dull Moment"? It's another great movie you might enjoy.

User: Could you tell me more about "Never a Dull Moment"?

Agent: Sure! "Never a Dull Moment" is a comedy film from the 60s. It's about an actor who gets m

In [14]:
# conversation with its metadata side by side
sample = profiles[1]
conv_meta = sample['info']['Conversation'][0]['conversation_1']
cid = conv_meta['conversation_id']

print(f"User: {sample['user_id']} | History: {sample['history_count']} movies")
print(f"Likes:    {[item_map.get(x, x) for x in conv_meta['user_likes']]}")
print(f"Dislikes: {[item_map.get(x, x) for x in conv_meta['user_dislikes']]}")
print(f"Rec'd:    {[item_map.get(x, x) for x in conv_meta['rec_item']]}")
print(f'\n{"="*80}\n')
print(convos[cid])

User: A2NBOL825B93OM | History: 38 movies
Likes:    ['The Lord of the Rings: The Fellowship of the Ring']
Dislikes: ['Driven VHS']
Rec'd:    ['The Lord Of The Rings: The Return Of The King']


User: Hello, I had a terrible experience with the movie "Driven VHS". The plot was completely improbable and the characters were unrealistic. I would not recommend it at all.

Agent: I'm sorry to hear that you didn't enjoy "Driven VHS". It can be disappointing when a movie doesn't meet our expectations. Is there anything else I can help you with?

User: Yes, I'm actually looking for a movie recommendation. Have you watched "The Lord of the Rings: The Fellowship of the Ring"?

Agent: Yes, I have come across "The Lord of the Rings: The Fellowship of the Ring". It seems to have positive reviews. However, based on your dislike for "Driven VHS", I would recommend "The Lord of the Rings: The Return of the King". It has received great reviews and is part of the same trilogy.

User: Thank you for the rec

In [15]:
# turn count distribution
turn_data = []
for cid, text in convos.items():
    turns = text.count('User:') + text.count('Agent:')
    turn_data.append({'conv_id': cid, 'turns': turns, 'chars': len(text)})

df_turns = pd.DataFrame(turn_data)
print(df_turns[['turns', 'chars']].describe())

print('\nTurn distribution:')
turn_dist = Counter(df_turns['turns'])
for t in sorted(turn_dist):
    bar = '█' * (turn_dist[t] // 10)
    print(f'  {t:3d} turns: {turn_dist[t]:5d} {bar}')

              turns         chars
count  10089.000000  10089.000000
mean      10.561304   1669.462385
std        2.428688    467.488133
min        4.000000    530.000000
25%        9.000000   1332.000000
50%       10.000000   1646.000000
75%       12.000000   1963.000000
max       22.000000   4792.000000

Turn distribution:
    4 turns:    47 ████
    5 turns:    20 ██
    6 turns:   671 ███████████████████████████████████████████████████████████████████
    7 turns:   151 ███████████████
    8 turns:  1622 ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    9 turns:   131 █████████████
   10 turns:  2936 ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

In [16]:
# dataset coverage
all_referenced = set(all_history)
for p in profiles:
    all_referenced.update(p['info']['user_might_like'])
    for j, conv in enumerate(p['info']['Conversation']):
        cd = conv[f'conversation_{j+1}']
        all_referenced.update(cd['user_likes'])
        all_referenced.update(cd['user_dislikes'])
        all_referenced.update(cd['rec_item'])

unreferenced = set(item_map.keys()) - all_referenced
print(f'Catalog: {len(item_map)} | Referenced: {len(all_referenced)} | Unreferenced: {len(unreferenced)}')

Catalog: 9687 | Referenced: 9687 | Unreferenced: 0


# Data Quality Deep Dive

In [17]:
# A) short names - garbage?
short = [(mid, n) for mid, n in item_map.items() if len(n) <= 3]
print(f'Short names (<=3 chars): {len(short)}')
for mid, n in short:
    print(f'  [{mid}] "{n}"')

Short names (<=3 chars): 26
  [B00005JKMQ] "Ali"
  [B00005JND5] "Ray"
  [B0007IF19K] "Ray"
  [B00005U8EL] "O"
  [B0019LY5IC] "W"
  [B00009MEC4] "May"
  [B003Q6D2B4] "Red"
  [B00008K77D] "Max"
  [B005TMXXX0] "Gog"
  [B00005MKKV] "Wit"
  [B00H9HZITU] "Her"
  [B001C47ZOW] "Tim"
  [6304701527] "Tim"
  [B00005QAPH] "8.5"
  [B005LAII1C] "Ted"
  [B001BMN35K] "XXy"
  [B009NNM9OA] "42"
  [B001CT8762] "Red"
  [B00BJ0RGCU] "Mud"
  [B00005JPUW] "War"
  [B0017S66JI] "Rec"
  [B00H4IR3JC] "Flu"
  [B00LM4A21A] "WER"
  [B00IKM5OYW] "Joe"
  [B004HO6I4M] "Rio"
  [B01EYF72MK] "McQ"


In [18]:
# B) long names - product listings
long_names = [(mid, n) for mid, n in item_map.items() if len(n) > 100]
print(f'Long names (>100 chars): {len(long_names)}')
for mid, n in sorted(long_names, key=lambda x: len(x[1]), reverse=True)[:10]:
    print(f'  [{len(n):3d}ch] {n[:120]}...')

Long names (>100 chars): 148
  [352ch] Nausicaa of the Valley of the Wind (1985) / REGION 6 DVD / Audio: English, Japanese, French, Simp. and Trad. Chinese, Th...
  [248ch] The Son's Room (2001) / REGION FREE NTSC DVD / Audio: French, Italian, Japanese, Chinese / Subtitles: English, Simp. and...
  [236ch] L'Eclisse [1962] / Region 2 PAL European Edition DVD / Language: Italian / Subtitles: English / Actors: Monica Vitti, Al...
  [216ch] The 100-Year-Old Man Who Climbed Out the Window and Disappeared ( Hundra&aring;ringen som klev ut genom f&ouml;nstret oc...
  [195ch] The Maltese Falcon (1931) &amp; Satan Met a Lady (1936) - Authentic Region 1 DVD from Warner Brothers starring Bette Dav...
  [194ch] The Premiere Frank Capra Collection (Mr. Smith Goes to Washington / It Happened One Night / You Can't Take It with You /...
  [185ch] The Grey Zone (2002) / REGION 2 PAL DVD / Audio: English / Actors: Daniel Benzali, Steve Buscemi, Mira Sorvino, David Ar...
  [185ch] Marilyn Monroe: The Dia

In [19]:
# C) HTML entities
html_ent = [(mid, n) for mid, n in item_map.items() if '&' in n and ';' in n]
print(f'HTML entities: {len(html_ent)}')
for mid, n in html_ent[:10]:
    print(f'  {n}')

HTML entities: 308
  Kate &amp; Leopold
  Benny &amp; Joon VHS
  Quatermass &amp; The Pit VHS
  A Boy &amp; His Dog
  A Boy &amp; His Dog VHS
  Star Trek - Deep Space Nine, Episodes 1 &amp; 2: The Emissary Pilot  VHS
  Hole in the Wall / Go &lsquo;Head On
  Tucker &amp; Dale vs. Evil
  Cowboys &amp; Aliens
  The Whistleblower [2010, Canada] DVD Starring Rachel Weisz &amp; Monica Bellucci


In [20]:
# D) duplicate movies - same film, different format/edition
def normalize_title(name):
    n = name.strip()
    for tag in [' VHS', ' [VHS]', ' DVD', ' [DVD]', ' Blu-ray', ' [Blu-ray]']:
        n = n.replace(tag, '')
    n = re.sub(r'\[.*?\]', '', n)
    n = re.sub(r'\((?!\d{4}\))[^)]*\)', '', n)
    n = re.sub(r'\s+', ' ', n).strip().lower()
    return n

title_groups = defaultdict(list)
for mid, name in item_map.items():
    title_groups[normalize_title(name)].append((mid, name))

dupes = {k: v for k, v in title_groups.items() if len(v) > 1}
print(f'Unique normalized: {len(title_groups)} | Duped titles: {len(dupes)} | Extra entries: {sum(len(v) for v in dupes.values()) - len(dupes)}')

for title, entries in sorted(dupes.items(), key=lambda x: len(x[1]), reverse=True)[:10]:
    print(f'\n  "{title}" ({len(entries)} versions):')
    for mid, name in entries:
        print(f'    [{mid}] {name}  (watched {movie_freq.get(mid, 0)}x)')

Unique normalized: 9307 | Duped titles: 348 | Extra entries: 380

  "journey to the center of the earth" (6 versions):
    [6305778256] Journey to the Center of the Earth  (watched 6x)
    [B0016Q2D5M]  Journey to the Center of the Earth  (watched 39x)
    [B00007JMD8] Journey to the Center of the Earth  (watched 34x)
    [6302098424] Journey to the Center of the Earth VHS  (watched 10x)
    [B0016MJ6HE] Journey to the Center of the Earth  (watched 6x)
    [B0017UAXJK] Journey to the Center of the Earth  (watched 3x)

  "gulliver's travels" (5 versions):
    [6304111835] Gulliver's Travels VHS  (watched 8x)
    [B00003ETJV] Gulliver's Travels  (watched 8x)
    [1572525959] Gulliver's Travels VHS  (watched 6x)
    [B00006D28S] Gulliver's Travels  (watched 6x)
    [B002ZG97WO] Gulliver's Travels  (watched 18x)

  "doctor who" (4 versions):
    [B00F37VHPM] Doctor Who  (watched 16x)
    [B002SZQC7K]  Doctor Who  (watched 8x)
    [B0016821YI]  Doctor Who  (watched 7x)
    [B002IW62FK]  Doc

In [21]:
# E) product noise categories
noise_patterns = {
    'Region codes': r'region\s*\d|region free',
    'Edition tags': r'edition|steelbook|collector|limited|special|widescreen|fullscreen|letterbox',
    'Disc info': r'\d+\s*disc|dual layer|single disc',
    'Language tags': r'mandarin|cantonese|hindi|spanish edition|french edition|chinese edition',
    'Format specs': r'thx|digitally|remastered|restored|unrated|uncut|extended cut|director.s cut',
    'Amazon/product': r'import|region \d|ntsc|pal|asin',
    'Year in title': r'\(\d{4}\)|\b(19|20)\d{2}\b',
    'Slashes (multi-pack)': r'/.+/',
}

noise_results = []
for label, pattern in noise_patterns.items():
    matches = [n for n in names if re.search(pattern, n, re.IGNORECASE)]
    noise_results.append({'Pattern': label, 'Count': len(matches), 'Example': matches[0] if matches else ''})

pd.DataFrame(noise_results)

,Pattern,Count,Example
0,Region codes,101,Sideways (Region 2)
1,Edition tags,127,Finding Nemo (Mandarin Chinese Edition) [2 DVDs]
2,Disc info,7,Lord of the Rings Return of the King 2 Disc Ex...
3,Language tags,20,Finding Nemo (Mandarin Chinese Edition) [2 DVDs]
4,Format specs,42,Daredevil (Director's Cut) / Elektra [DVD]
5,Amazon/product,167,Sideways (Region 2)
6,Year in title,307,Timeline (2003) (Widescreen)
7,Slashes (multi-pack),165,Pirates of the Golden Age Movie Collection: (A...


In [22]:
# F) conversation quality
no_rec = [cid for cid, text in convos.items() if 'recommend' not in text.lower() and 'suggest' not in text.lower()]
print(f'No recommendation language: {len(no_rec)}')

# quoted strings in dialogues vs catalog
all_quoted = []
for cid, text in convos.items():
    all_quoted.extend(re.findall(r'"([^"]+)"', text))

unique_quoted = set(all_quoted)
catalog_names = set(item_map.values())
in_catalog = unique_quoted & catalog_names
not_in_catalog = unique_quoted - catalog_names
print(f'Quoted strings: {len(unique_quoted)} unique | In catalog: {len(in_catalog)} | Not: {len(not_in_catalog)}')

# sample non-matching (filter out parse artifacts)
real_mismatches = [q for q in sorted(not_in_catalog) if not q.startswith('\n') and len(q) > 3 and len(q) < 80]
print(f'\nSample non-matching movie references:')
for q in real_mismatches[:15]:
    print(f'  "{q}"')

No recommendation language: 4
Quoted strings: 14510 unique | In catalog: 5365 | Not: 9145

Sample non-matching movie references:
  " '71 2014"
  " Here's a review from another user: "
  " a try.

Agent: Great choice! Enjoy watching "
  " as well. Here's my review: "
  " comes from Rob Marshall (Chicago, Memoirs of a Geisha). Unfortunately, \"
  " is better (cause all they did was \"
  " too. Here's my review: "
  " were not as gory looking as I had anticipated."
  " while shooting film of the 'monsters' destroying South Park..."
  "'71 2014."
  "* * * * 2005. Written and directed by John Turturro..."
  ". It was great on the big screen and also on my 60"
  "10 Cloverfield Lane."
  "10 Things I Hate About You"
  "10 Things I Hate About You."


In [23]:
# G) data integrity checks
already_watched_recs = 0
like_not_in_history = 0

for p in profiles:
    history = set(p['info']['history_interaction'])
    for j, conv in enumerate(p['info']['Conversation']):
        cd = conv[f'conversation_{j+1}']
        for rec in cd['rec_item']:
            if rec in history:
                already_watched_recs += 1
        for like in cd['user_likes']:
            if like not in history:
                like_not_in_history += 1

might_like_already_watched = 0
for p in profiles:
    history = set(p['info']['history_interaction'])
    for mid in p['info']['user_might_like']:
        if mid in history:
            might_like_already_watched += 1

# case-only duplicates
name_lower_map = defaultdict(list)
for mid, name in item_map.items():
    name_lower_map[name.lower()].append((mid, name))
case_dupes = {k: v for k, v in name_lower_map.items() if len(v) > 1}

print(f'Rec already watched:     {already_watched_recs}')
print(f'Like not in history:     {like_not_in_history}')
print(f'MightLike in history:    {might_like_already_watched}')
print(f'Case-only dupes:         {len(case_dupes)}')

for title, entries in sorted(case_dupes.items(), key=lambda x: len(x[1]), reverse=True)[:5]:
    print(f'  {[name for _, name in entries]}')

Rec already watched:     14
Like not in history:     9
MightLike in history:    14
Case-only dupes:         168
  ['Journey to the Center of the Earth', 'Journey to the Center of the Earth', 'Journey to the Center of the Earth', 'Journey to the Center of the Earth']
  ['Mansfield Park VHS', 'Mansfield Park VHS', 'Mansfield Park VHS']
  ["Gulliver's Travels", "Gulliver's Travels", "Gulliver's Travels"]
  [' Doctor Who', ' Doctor Who', ' Doctor Who']
  ['Terminator 3: Rise of the Machines', 'Terminator 3: Rise of the Machines']


In [24]:
# H) summary - all issues as a dataframe
issues = [
    ('VHS in name', vhs),
    ('Duplicate titles (format/edition)', sum(len(v) for v in dupes.values()) - len(dupes)),
    ('HTML entities', len(html_ent)),
    ('Case-only dupes', len(case_dupes)),
    ('DVD in name', dvd),
    ('Long names (>100ch)', len(long_names)),
    ('Blu-ray in name', bluray),
    ('Short names (<=3ch)', len(short)),
    ('Rec already watched', already_watched_recs),
    ('MightLike already watched', might_like_already_watched),
    ('Like not in history', like_not_in_history),
]

df_issues = pd.DataFrame(issues, columns=['Issue', 'Count'])
df_issues = df_issues.sort_values('Count', ascending=False).reset_index(drop=True)
df_issues['% of catalog'] = (df_issues['Count'] / len(item_map) * 100).round(1)
df_issues

,Issue,Count,% of catalog
0,VHS in name,2927,30.2
1,Duplicate titles (format/edition),380,3.9
2,HTML entities,308,3.2
3,Case-only dupes,168,1.7
4,DVD in name,167,1.7
5,Long names (>100ch),148,1.5
6,Blu-ray in name,128,1.3
7,Short names (<=3ch),26,0.3
8,Rec already watched,14,0.1
9,MightLike already watched,14,0.1


In [25]:
# search for any movie and see all its variants
def find_variants(query):
    query_lower = query.lower()
    matches = [(mid, name) for mid, name in item_map.items() if query_lower in name.lower()]
    print(f'"{query}" → {len(matches)} entries:\n')
    for mid, name in matches:
        watched = movie_freq.get(mid, 0)
        liked = like_freq.get(mid, 0)
        print(f'  [{mid}] {name}')
        print(f'         watched: {watched}x | liked: {liked}x')
    return matches

# try these:
find_variants("alice in wonderland")
# find_variants("star wars")
# find_variants("blade runner")
# find_variants("lord of the rings")
# find_variants("titanic")
# find_variants("the sixth sense")

"alice in wonderland" → 4 entries:

  [6300274268] Alice in Wonderland Walt Disney Masterpiece Collection  VHS
         watched: 54x | liked: 4x
  [B001HN6940]  Alice in Wonderland [Blu-ray]
         watched: 46x | liked: 4x
  [B0030U1TFW] Alice in Wonderland (1933)
         watched: 3x | liked: 0x
  [6305372837] Alice in Wonderland VHS
         watched: 8x | liked: 0x


[('6300274268', 'Alice in Wonderland Walt Disney Masterpiece Collection  VHS'),
 ('B001HN6940', ' Alice in Wonderland [Blu-ray]'),
 ('B0030U1TFW', 'Alice in Wonderland (1933)'),
 ('6305372837', 'Alice in Wonderland VHS')]

In [26]:
# ---- FULL NUANCE SCAN: find ALL types of messy movie names ----

import html as html_lib
from difflib import SequenceMatcher

# 1) Strip everything to get a "core title"
def core_title(name):
    n = html_lib.unescape(name).strip()
    # remove format tags
    for tag in ['VHS', 'DVD', 'Blu-ray', 'Blu-Ray', 'NTSC', 'PAL']:
        n = re.sub(rf'\b{tag}\b', '', n, flags=re.IGNORECASE)
    # remove bracketed content: [VHS], [Blu-ray], [2 DVDs], [Region 2], etc.
    n = re.sub(r'\[.*?\]', '', n)
    # remove parenthesized non-year content: (Widescreen), (Mandarin Chinese Edition)
    n = re.sub(r'\((?!\d{4}\))[^)]*\)', '', n)
    # remove trailing year like (1933)
    n = re.sub(r'\(\d{4}\)\s*$', '', n)
    # remove "region free", "region 1", etc.
    n = re.sub(r'region\s*(free|\d)', '', n, flags=re.IGNORECASE)
    # remove edition/steelbook/import/limited etc.
    n = re.sub(r'\b(edition|steelbook|collector|limited|import|widescreen|fullscreen|letterbox|unrated|uncut|remastered|digitally|restored|THX)\b', '', n, flags=re.IGNORECASE)
    # remove disc info
    n = re.sub(r'\d+\s*disc\w*', '', n, flags=re.IGNORECASE)
    # collapse whitespace, strip punctuation edges
    n = re.sub(r'\s+', ' ', n).strip().strip('- /:,.')
    return n.lower()

# group all movies by core title
core_groups = defaultdict(list)
for mid, name in item_map.items():
    core_groups[core_title(name)].append((mid, name))

# find groups with multiple entries
multi = {k: v for k, v in core_groups.items() if len(v) > 1}
print(f'Total movies: {len(item_map)}')
print(f'Unique core titles: {len(core_groups)}')
print(f'Core titles with variants: {len(multi)}')
print(f'Total variant entries: {sum(len(v) for v in multi.values())}')
print(f'Wasted entries (duplicates): {sum(len(v) for v in multi.values()) - len(multi)}')

# categorize WHY they're duplicates
reasons = {
    'format_diff': [],      # VHS vs DVD vs Blu-ray
    'case_diff': [],        # same name different case
    'edition_diff': [],     # steelbook, widescreen, etc.
    'year_diff': [],        # (1933) vs no year vs (2010)
    'language_diff': [],    # Mandarin, Español, etc.
    'leading_space': [],    # " Alice" vs "Alice"
    'same_exact_name': [],  # identical name, different ASIN
    'other': [],
}

for core, entries in multi.items():
    raw_names = [name for _, name in entries]
    lower_names = [n.lower().strip() for n in raw_names]

    has_vhs = any('vhs' in n.lower() for n in raw_names)
    has_dvd = any('dvd' in n.lower() for n in raw_names)
    has_blu = any('blu' in n.lower() for n in raw_names)
    has_format_mix = sum([has_vhs, has_dvd, has_blu]) > 0 and len(set(lower_names)) > 1

    if any(n.startswith(' ') for n in raw_names):
        reasons['leading_space'].append((core, entries))
    elif len(set(raw_names)) == 1:
        reasons['same_exact_name'].append((core, entries))
    elif len(set(lower_names)) == 1 and len(set(raw_names)) > 1:
        reasons['case_diff'].append((core, entries))
    elif has_format_mix:
        reasons['format_diff'].append((core, entries))
    elif any(re.search(r'\(\d{4}\)', n) for n in raw_names) and any(not re.search(r'\(\d{4}\)', n) for n in raw_names):
        reasons['year_diff'].append((core, entries))
    elif any(re.search(r'mandarin|espa|latin|french edition|chinese|hindi|cantonese', n, re.I) for n in raw_names):
        reasons['language_diff'].append((core, entries))
    elif any(re.search(r'edition|steelbook|collector|widescreen|extended|special', n, re.I) for n in raw_names):
        reasons['edition_diff'].append((core, entries))
    else:
        reasons['other'].append((core, entries))

print(f'\n--- WHY are they duplicates? ---')
for reason, items in sorted(reasons.items(), key=lambda x: len(x[1]), reverse=True):
    print(f'\n  {reason}: {len(items)} groups')
    for core, entries in items[:3]:
        names = [name for _, name in entries]
        print(f'    "{core}" → {names}')

Total movies: 9687
Unique core titles: 9292
Core titles with variants: 358
Total variant entries: 753
Wasted entries (duplicates): 395

--- WHY are they duplicates? ---

  format_diff: 188 groups
    "white christmas" → ['White Christmas', 'White Christmas VHS']
    "frailty" → ['Frailty', 'Frailty VHS']
    "sexy beast" → ['Sexy Beast', 'Sexy Beast VHS']

  same_exact_name: 137 groups
    "terminator 3: rise of the machines" → ['Terminator 3: Rise of the Machines', 'Terminator 3: Rise of the Machines']
    "somewhere tomorrow" → ['Somewhere Tomorrow', 'Somewhere Tomorrow']
    "lara croft: tomb raider" → ['Lara Croft: Tomb Raider', 'Lara Croft: Tomb Raider']

  leading_space: 17 groups
    "constantine" → ['Constantine', 'Constantine', ' Constantine [Blu-ray] [Blu-ray] (2008)']
    "doctor who" → ['Doctor Who', ' Doctor Who', ' Doctor Who', ' Doctor Who']
    "where the wild things are" → ['Where the Wild Things Are', ' Where the Wild Things Are [Blu-ray]']

  other: 9 groups
    "two

In [ ]:
# ---- IMPACT: how much do duplicates affect user histories? ----

# for each user, check how many "different" movies in their history
# are actually the same movie after normalization
users_with_dupe_history = 0
total_phantom_movies = 0

for p in profiles:
    history_ids = p['info']['history_interaction']
    core_titles_seen = defaultdict(list)
    for mid in history_ids:
        name = item_map.get(mid, mid)
        ct = core_title(name)
        core_titles_seen[ct].append(name)

    dupes_in_history = {k: v for k, v in core_titles_seen.items() if len(v) > 1}
    if dupes_in_history:
        users_with_dupe_history += 1
        total_phantom_movies += sum(len(v) - 1 for v in dupes_in_history.values())

print(f'Users with duplicate movies in history: {users_with_dupe_history} / {len(profiles)} ({users_with_dupe_history/len(profiles)*100:.1f}%)')
print(f'Total phantom entries (same movie counted twice): {total_phantom_movies}')

# show worst examples
print(f'\nWorst cases:')
worst = []
for p in profiles:
    history_ids = p['info']['history_interaction']
    core_titles_seen = defaultdict(list)
    for mid in history_ids:
        name = item_map.get(mid, mid)
        core_titles_seen[core_title(name)].append(name)
    dupes = {k: v for k, v in core_titles_seen.items() if len(v) > 1}
    if dupes:
        worst.append((p['user_id'], len(p['info']['history_interaction']), dupes))

worst.sort(key=lambda x: sum(len(v) for v in x[2].values()), reverse=True)
for uid, hist_count, dupes in worst[:5]:
    print(f'\n  User {uid} (history: {hist_count} movies, {len(dupes)} duplicated titles):')
    for core, names in dupes.items():
        print(f'    "{core}" → {names}')

In [27]:
# ---- FULL LIST: every duplicate group with all its variants ----
# sorted by total watch count (most impactful first)

print(f'ALL {len(multi)} duplicate groups:\n')
ranked = []
for core, entries in multi.items():
    total_watched = sum(movie_freq.get(mid, 0) for mid, _ in entries)
    ranked.append((core, entries, total_watched))

ranked.sort(key=lambda x: x[2], reverse=True)

for i, (core, entries, total_watched) in enumerate(ranked[:50]):
    print(f'{i+1:3d}. "{core}" ({len(entries)} variants, {total_watched} total watches)')
    for mid, name in entries:
        w = movie_freq.get(mid, 0)
        print(f'      [{mid}] {name}  ({w}x)')
    print()

ALL 358 duplicate groups:

  1. "the sixth sense" (2 variants, 345 total watches)
      [630574999X] THE SIXTH SENSE  (172x)
      [630573240X] The Sixth Sense VHS  (173x)

  2. "the lord of the rings: the return of the king" (2 variants, 276 total watches)
      [B00005JKZY] The Lord of the Rings: The Return of the King  (161x)
      [B00Z9YZRVE] The Lord Of The Rings: The Return Of The King  (115x)

  3. "king kong" (2 variants, 256 total watches)
      [B01AHWLX2O] King Kong (The Huntsman: Winter's War Fandango Cash Version) [Blu-ray]  (110x)
      [B00005JO20] King Kong  (146x)

  4. "terminator 3: rise of the machines" (2 variants, 238 total watches)
      [B000E1MTYU] Terminator 3: Rise of the Machines  (118x)
      [B00005JM0B] Terminator 3: Rise of the Machines  (120x)

  5. "guardians of the galaxy" (3 variants, 220 total watches)
      [B00PY4Q9OS] Guardians of the Galaxy  (78x)
      [B00Q0G2VXM] Guardians Of The Galaxy Region Free  (75x)
      [B00R8GUXPG] Guardians of the 

In [34]:
# ---- WHAT DID THE NORMALIZER MISS? ----
# check for near-matches using fuzzy similarity

from difflib import SequenceMatcher

# get all unique core titles (only singles - not already caught as dupes)
singles = {k: v[0] for k, v in core_groups.items() if len(v) == 1}

# also handle "The X" vs "X, The" pattern
def strip_article(title):
    """'terminator, the' -> 'terminator' | 'the terminator' -> 'terminator'"""
    t = title.strip()
    if t.startswith('the '):
        t = t[4:]
    if t.endswith(', the'):
        t = t[:-5]
    if t.startswith('a '):
        t = t[2:]
    if t.endswith(', a'):
        t = t[:-3]
    return t.strip()

# first: check article rearrangement (fast, exact)
article_groups = defaultdict(list)
for core, (mid, name) in singles.items():
    article_groups[strip_article(core)].append((core, mid, name))

article_dupes = {k: v for k, v in article_groups.items() if len(v) > 1}
print(f'"The X" vs "X, The" duplicates missed by normalizer: {len(article_dupes)}')
for stripped, entries in sorted(article_dupes.items(), key=lambda x: len(x[1]), reverse=True)[:20]:
    print(f'\n  "{stripped}":')
    for core, mid, name in entries:
        print(f'    [{mid}] {name}  (watched {movie_freq.get(mid, 0)}x)')

"The X" vs "X, The" duplicates missed by normalizer: 35

  "christmas carol":
    [6303824358] A Christmas Carol VHS  (watched 28x)
    [6301967275] Christmas Carol VHS  (watched 14x)

  "christmas story":
    [6301966554] A Christmas Story VHS  (watched 64x)
    [B0021GYDAE] Christmas Story ( Joulutarina ) ( En Julber&auml;ttelse ) [ NON-USA FORMAT, PAL, Reg.2 Import - Finland ]  (watched 4x)

  "knight's tale":
    [B00000F4ZY] A Knight's Tale  (watched 53x)
    [B000IHYBT6] Knight's Tale  (watched 52x)

  "curse of frankenstein":
    [B00006G8JZ] The Curse of Frankenstein  (watched 18x)
    [6302814669] Curse of Frankenstein VHS  (watched 19x)

  "fish called wanda":
    [6304196784] Fish Called Wanda VHS  (watched 41x)
    [6305161879] A Fish Called Wanda  (watched 39x)

  "descendants":
    [B004UXUX4Q] The Descendants  (watched 46x)
    [B00X797NJW] Descendants  (watched 7x)

  "secret in their eyes":
    [B0036TGSJE] The Secret in Their Eyes  (watched 18x)
    [B0182W7KBY] Secre

In [32]:
# ---- IMPROVED NORMALIZER v2: punctuation + articles ----
# the original core_title missed ":"/" "/"-" differences and "The X" vs "X"

def core_title_v2(name):
    n = html_lib.unescape(name).strip()
    for tag in ['VHS', 'DVD', 'Blu-ray', 'Blu-Ray', 'NTSC', 'PAL']:
        n = re.sub(rf'\b{tag}\b', '', n, flags=re.IGNORECASE)
    n = re.sub(r'\[.*?\]', '', n)
    n = re.sub(r'\((?!\d{4}\))[^)]*\)', '', n)
    n = re.sub(r'\(\d{4}\)\s*$', '', n)
    n = re.sub(r'region\s*(free|\d)', '', n, flags=re.IGNORECASE)
    n = re.sub(r'\b(edition|steelbook|collector|limited|import|widescreen|fullscreen|letterbox|unrated|uncut|remastered|digitally|restored|THX)\b', '', n, flags=re.IGNORECASE)
    n = re.sub(r'\d+\s*disc\w*', '', n, flags=re.IGNORECASE)
    # NEW: normalize punctuation (: - , /) to space
    n = re.sub(r'[:\-,/]', ' ', n)
    # NEW: normalize ampersand
    n = n.replace('&', 'and')
    n = re.sub(r'\s+', ' ', n).strip().strip('- /:,.')
    n = n.lower()
    # NEW: strip leading articles
    n = re.sub(r'^(the|a|an)\s+', '', n)
    # NEW: strip trailing ", the" / ", a"
    n = re.sub(r',\s*(the|a|an)\s*$', '', n)
    return n.strip()

# regroup with v2
core_groups_v2 = defaultdict(list)
for mid, name in item_map.items():
    core_groups_v2[core_title_v2(name)].append((mid, name))

multi_v2 = {k: v for k, v in core_groups_v2.items() if len(v) > 1}
print(f'--- core_title v1 vs v2 ---')
print(f'v1: {len(core_groups)} unique → {len(multi)} duplicate groups')
print(f'v2: {len(core_groups_v2)} unique → {len(multi_v2)} duplicate groups')
print(f'v2 caught {len(multi_v2) - len(multi)} MORE groups')

# show what's new (in v2 but not v1)
v1_cores = set(core_title(name) for name in item_map.values())
new_dupes = []
for core_v2, entries in multi_v2.items():
    v1_keys = set(core_title(name) for _, name in entries)
    if all(len(core_groups.get(k, [])) <= 1 for k in v1_keys):
        total = sum(movie_freq.get(mid, 0) for mid, _ in entries)
        new_dupes.append((core_v2, entries, total))

new_dupes.sort(key=lambda x: x[2], reverse=True)
print(f'\n--- NEW duplicates caught by v2 (top 25): ---\n')
for core_v2, entries, total in new_dupes[:25]:
    print(f'  "{core_v2}" ({len(entries)} variants, {total} total watches)')
    for mid, name in entries:
        w = movie_freq.get(mid, 0)
        print(f'    [{mid}] {name}  ({w}x)')
    print()

--- core_title v1 vs v2 ---
v1: 9292 unique → 358 duplicate groups
v2: 9232 unique → 411 duplicate groups
v2 caught 53 MORE groups

--- NEW duplicates caught by v2 (top 25): ---

  "resident evil apocalypse" (2 variants, 190 total watches)
    [0767834739] Resident Evil: Apocalypse  (97x)
    [B000BYRCR4] Resident Evil - Apocalypse  (93x)

  "truman show" (2 variants, 126 total watches)
    [6305183023] Truman Show VHS  (63x)
    [B00000IRED] The Truman Show VHS  (63x)

  "knight's tale" (2 variants, 105 total watches)
    [B00000F4ZY] A Knight's Tale  (53x)
    [B000IHYBT6] Knight's Tale  (52x)

  "star wars episode vi return of the jedi" (2 variants, 93 total watches)
    [6301773578] Star Wars - Episode VI, Return of the Jedi VHS  (47x)
    [B00076SCPW] Star Wars, Episode VI: Return of the Jedi  (46x)

  "jackass the movie" (2 variants, 82 total watches)
    [B00008972B] Jackass: The Movie  (42x)
    [B00008972L] Jackass - The Movie VHS  (40x)

  "ben hur" (2 variants, 81 total watc

In [33]:
# ---- WORD-SET JACCARD: catch remaining stragglers v2 missed ----
# idea: two titles are the same movie if their WORD SETS overlap heavily
# but different sequels ("Scary Movie 2" vs "Scary Movie 3") differ by numbers
# so we keep numbers as distinguishing tokens

singles_v2 = {k: v[0] for k, v in core_groups_v2.items() if len(v) == 1}
print(f'Singletons after v2: {len(singles_v2)} — checking word-set overlap...\n')

def title_words(t):
    """split into word set, keeping numbers as-is"""
    return set(re.findall(r'\w+', t.lower()))

# only check frequent singletons (watched >= 15 total)
freq_singles = [(core, mid, name) for core, (mid, name) in singles_v2.items()
                if movie_freq.get(mid, 0) >= 15]
print(f'Frequent singletons to check: {len(freq_singles)}')

jaccard_dupes = []
for i, (ca, ma, na) in enumerate(freq_singles):
    wa = title_words(ca)
    if len(wa) < 2:  # skip single-word titles (too ambiguous)
        continue
    for j, (cb, mb, nb) in enumerate(freq_singles):
        if i >= j:
            continue
        wb = title_words(cb)
        if len(wb) < 2:
            continue
        overlap = wa & wb
        union = wa | wb
        jaccard = len(overlap) / len(union)
        # require high jaccard AND at least 3 shared words
        if jaccard >= 0.75 and len(overlap) >= 3:
            jaccard_dupes.append((jaccard, len(overlap), ca, na, ma, cb, nb, mb))

jaccard_dupes.sort(key=lambda x: (x[0], x[1]), reverse=True)
print(f'Found {len(jaccard_dupes)} word-overlap matches:\n')

for jac, noverlap, ca, na, ma, cb, nb, mb in jaccard_dupes:
    wa = movie_freq.get(ma, 0)
    wb = movie_freq.get(mb, 0)
    print(f'  Jaccard={jac:.0%} ({noverlap} shared words)')
    print(f'    A: {na}  ({wa}x)')
    print(f'    B: {nb}  ({wb}x)')
    print()

# ---- FINAL SUMMARY ----
print('\n' + '=' * 55)
print('FINAL DUPLICATION DETECTION SUMMARY (v2 normalizer)')
print('=' * 55)
print(f'  v2 normalizer groups:          {len(multi_v2):4d} groups')
print(f'  Word-set Jaccard stragglers:   {len(jaccard_dupes):4d} pairs')
print(f'  {"─"*45}')
total_dupe_movies = sum(len(v) for v in multi_v2.values()) - len(multi_v2)
print(f'  Total duplicate entries:       {total_dupe_movies:4d}')
print(f'  Catalog size:                  {len(item_map):4d}')
print(f'  Truly unique movies (v2):      {len(core_groups_v2) - len(jaccard_dupes):4d}')
print(f'\n  Improvement over v1: {len(multi_v2) - len(multi)} more groups caught')

Singletons after v2: 8821 — checking word-set overlap...

Frequent singletons to check: 2504
Found 13 word-overlap matches:

  Jaccard=86% (6 shared words)
    A: The Hobbit: The Battle of the Five Armies [DVD] [2015]  (51x)
    B: Hobbit 3: The Battle of the Five Armies  (45x)

  Jaccard=83% (5 shared words)
    A: Godzilla: King of the Monsters VHS  (19x)
    B: Gojira / Godzilla, King of the Monsters  (19x)

  Jaccard=80% (4 shared words)
    A: The Best Exotic Marigold Hotel  (55x)
    B: The Second Best Exotic Marigold Hotel [DVD] [2015]  (16x)

  Jaccard=80% (4 shared words)
    A: The Best Exotic Marigold Hotel  (55x)
    B: Best Exotic Marigold Hotel 2  (16x)

  Jaccard=78% (7 shared words)
    A: Star Trek The Next Generation - The Complete Fifth Season  (17x)
    B: Star Trek The Next Generation - The Complete Third Season  (18x)

  Jaccard=75% (6 shared words)
    A: Buffy the Vampire Slayer - The Complete Fifth Season  (22x)
    B: Buffy the Vampire Slayer - The Complete Th